# 🚀 Proyecto Final: Sistema de Gestión de Paquetería Inteligente


## 🧩 Descripción General
Se debe implementar un sistema de gestión de envíos para una empresa de mensajería y logística que permita registrar paquetes, asociarlos a clientes, llevar el control de su estado, y calcular costos según el tipo de envío. El sistema debe incluir relaciones entre clases (composición, agregación, asociación), uso de métodos mágicos, herencia, encapsulación, y polimorfismo.

### 🎯 Objetivo del Proyecto
- Evaluar los siguientes conceptos:

- Clases y objetos

- Atributos públicos, protegidos y privados

- Métodos de instancia, clase y estáticos

- Métodos mágicos (__str__, __init__, etc.)

- Encapsulación con validaciones (getters/setters)

- Asociación, agregación y composición

- Herencia y polimorfismo

- Buenas prácticas (modularidad, documentación)

### ESTRUCTURA E LAS CLASES

| Clase           | Rol                                                                 |
|----------------|----------------------------------------------------------------------|
| Cliente         | Representa a un usuario que envía o recibe paquetes                  |
| Paquete         | Clase base para paquetes, incluye atributos comunes y métodos mágicos |
| PaqueteExpress  | Hereda de Paquete, incluye recargo por urgencia                      |
| PaqueteEstandar | Hereda de Paquete, sin recargo                                       |
| Direccion       | Composición con Cliente (cada cliente tiene una dirección)           |
| Envio           | Agrega un paquete y se asocia a un cliente                           |
| SistemaEnvios   | Administra clientes y envíos                                         |


### 🧪 Requisitos funcionales
1. Registrar clientes y asociarles una dirección.

2. Crear paquetes (express o estándar).

3. Asignar paquetes a envíos y clientes.

4. Calcular el precio de cada envío.

5. Mostrar listado de envíos con información detallada (polimorfismo).

6. Validar datos (precio, peso, nombre del cliente, etc.).

7. Mostrar reporte de todos los envíos realizados.

8. Agregar seguimiento del paquete (pendiente, en tránsito, entregado) usando métodos set_estado().

9. Guardar la información en un archivo .txt o .json.

10. Mostrar totales de ventas por tipo de envío (estándar vs express).

11. Diseñar un menú de opciones para registrar clientes/envíos de forma interactiva.



In [ ]:

import json
from datetime import datetime
from abc import ABC, abstractmethod


class Direccion:
    """Clase que representa una dirección (Composición con Cliente)"""
    
    def __init__(self, calle, ciudad, codigo_postal):
        self.__calle = calle
        self.__ciudad = ciudad
        self.__codigo_postal = codigo_postal
    

    def get_calle(self):
        return self.__calle
    
    def get_ciudad(self):
        return self.__ciudad
    
    def get_codigo_postal(self):
        return self.__codigo_postal
    
    
    def set_calle(self, calle):
        if len(calle.strip()) > 0:
            self.__calle = calle
        else:
            raise ValueError("La calle no puede estar vacía")
    
    def set_ciudad(self, ciudad):
        if len(ciudad.strip()) > 0:
            self.__ciudad = ciudad
        else:
            raise ValueError("La ciudad no puede estar vacía")
    
    def set_codigo_postal(self, codigo):
        if isinstance(codigo, str) and len(codigo) >= 5:
            self.__codigo_postal = codigo
        else:
            raise ValueError("Código postal debe tener al menos 5 caracteres")
    
    def __str__(self):
        return f"{self.__calle}, {self.__ciudad} - {self.__codigo_postal}"


class Cliente:
    """Clase que representa un cliente del sistema"""
    
    _contador_clientes = 0
    
    def __init__(self, nombre, telefono, email, direccion):
        if not self._validar_nombre(nombre):
            raise ValueError("Nombre debe tener al menos 2 caracteres")
        if not self._validar_email(email):
            raise ValueError("Email inválido")
        
        
        self.__id_cliente = self._generar_id()
        self.__nombre = nombre
        self.__telefono = telefono
        self.__email = email

        self.__direccion = direccion
        
        Cliente._contador_clientes += 1
    
    @classmethod
    def _generar_id(cls):
        """Genera ID único para cliente"""
        cls._contador_clientes += 1
        return f"CLI{cls._contador_clientes:03d}"
    
    @staticmethod
    def _validar_nombre(nombre):
        """Valida que el nombre tenga al menos 2 caracteres"""
        return len(nombre.strip()) >= 2
    
    @staticmethod
    def _validar_email(email):
        """Validación básica de email"""
        return "@" in email and "." in email
    
    
    def get_id(self):
        return self.__id_cliente
    
    def get_nombre(self):
        return self.__nombre
    
    def get_telefono(self):
        return self.__telefono
    
    def get_email(self):
        return self.__email
    
    def get_direccion(self):
        return self.__direccion
    
    def set_telefono(self, telefono):
        self.__telefono = telefono
    
    def set_email(self, email):
        if self._validar_email(email):
            self.__email = email
        else:
            raise ValueError("Email inválido")
    
    @classmethod
    def total_clientes(cls):
        return cls._contador_clientes
    
    def __str__(self):
        return f"Cliente {self.__id_cliente}: {self.__nombre} ({self.__email})"
    
    def __eq__(self, otro):
        return self.__id_cliente == otro.__id_cliente


class Paquete(ABC):
    """Clase base abstracta para paquetes"""
    
    ESTADOS_VALIDOS = ["pendiente", "en_transito", "entregado", "devuelto"]
    
    def __init__(self, descripcion, peso, valor_declarado):
        
        if peso <= 0:
            raise ValueError("El peso debe ser mayor a 0")
        if valor_declarado < 0:
            raise ValueError("El valor declarado no puede ser negativo")
        
        self._descripcion = descripcion
        self._peso = peso
        self._valor_declarado = valor_declarado
        self._estado = "pendiente"
        self._fecha_creacion = datetime.now()
    
    
    def get_descripcion(self):
        return self._descripcion
    
    def get_peso(self):
        return self._peso
    
    def get_valor_declarado(self):
        return self._valor_declarado
    
    def get_estado(self):
        return self._estado
    
    def get_fecha_creacion(self):
        return self._fecha_creacion
    

    def set_estado(self, estado):
        if estado in self.ESTADOS_VALIDOS:
            self._estado = estado
            print(f"Estado del paquete actualizado a: {estado}")
        else:
            raise ValueError(f"Estado inválido. Debe ser uno de: {self.ESTADOS_VALIDOS}")
    
    @abstractmethod
    def calcular_precio(self):
        """Método abstracto que debe implementar cada subclase"""
        pass
    
    @staticmethod
    def precio_por_peso(peso):
        """Calcula precio base por peso"""
        return peso * 2000  # $2000 por kg
    
    def __str__(self):
        return f"Paquete: {self._descripcion} ({self._peso}kg) - Estado: {self._estado}"


class PaqueteEstandar(Paquete):
    """Paquete estándar sin recargo"""
    
    def __init__(self, descripcion, peso, valor_declarado):
        super().__init__(descripcion, peso, valor_declarado)
        self._tipo = "Estándar"
    
    def calcular_precio(self):
        """Precio base sin recargo"""
        precio_base = self.precio_por_peso(self._peso)
        seguro = self._valor_declarado * 0.01
        return precio_base + seguro
    
    def get_tipo(self):
        return self._tipo
    
    def __str__(self):
        return f"Paquete Estándar: {self._descripcion} ({self._peso}kg) - ${self.calcular_precio():.0f}"

class PaqueteExpress(Paquete):
    """Paquete express con recargo por urgencia"""
    
    def __init__(self, descripcion, peso, valor_declarado, recargo_urgencia=0.5):
        super().__init__(descripcion, peso, valor_declarado)
        self._tipo = "Express"
        self._recargo_urgencia = recargo_urgencia  
    
    def calcular_precio(self):
        """Precio base + recargo por urgencia"""
        precio_base = self.precio_por_peso(self._peso)
        seguro = self._valor_declarado * 0.01
        recargo = precio_base * self._recargo_urgencia
        return precio_base + seguro + recargo
    
    def get_tipo(self):
        return self._tipo
    
    def get_recargo_urgencia(self):
        return self._recargo_urgencia
    
    def __str__(self):
        return f"Paquete Express: {self._descripcion} ({self._peso}kg) - ${self.calcular_precio():.0f} (Recargo: {self._recargo_urgencia*100}%)"


class Envio:
    """Clase que representa un envío (Agregación con Paquete, Asociación con Cliente)"""
    
    _contador_envios = 0
    
    def __init__(self, cliente_origen, cliente_destino, paquete):
        if not isinstance(paquete, Paquete):
            raise ValueError("Debe proporcionar un objeto Paquete válido")
        
        
        Envio._contador_envios += 1
        self.__id_envio = f"ENV{self._contador_envios:04d}"
        
        
        self.__cliente_origen = cliente_origen
        self.__cliente_destino = cliente_destino
    
        self.__paquete = paquete
        
        self.__fecha_envio = datetime.now()
        self.__precio_total = paquete.calcular_precio()
    
    
    def get_id_envio(self):
        return self.__id_envio
    
    def get_cliente_origen(self):
        return self.__cliente_origen
    
    def get_cliente_destino(self):
        return self.__cliente_destino
    
    def get_paquete(self):
        return self.__paquete
    
    def get_fecha_envio(self):
        return self.__fecha_envio
    
    def get_precio_total(self):
        return self.__precio_total
    
    def actualizar_estado_paquete(self, nuevo_estado):
        """Actualiza el estado del paquete asociado"""
        self.__paquete.set_estado(nuevo_estado)
    
    @classmethod
    def total_envios(cls):
        return cls._contador_envios
    
    def __str__(self):
        return f"Envío {self.__id_envio}: {self.__cliente_origen.get_nombre()} → {self.__cliente_destino.get_nombre()} | ${self.__precio_total:.0f}"


class SistemaEnvios:
    """Sistema principal que administra clientes y envíos"""
    
    def __init__(self):
        self.__clientes = []
        self.__envios = []
    
    def registrar_cliente(self, nombre, telefono, email, calle, ciudad, codigo_postal):
        """Registra un nuevo cliente con su dirección"""
        try:
            direccion = Direccion(calle, ciudad, codigo_postal)
            cliente = Cliente(nombre, telefono, email, direccion)
            self.__clientes.append(cliente)
            print(f"Cliente {cliente.get_nombre()} registrado exitosamente")
            return cliente
        except ValueError as e:
            print(f"Error al registrar cliente: {e}")
            return None
    
    def crear_paquete_estandar(self, descripcion, peso, valor_declarado):
        """Crea un paquete estándar"""
        try:
            return PaqueteEstandar(descripcion, peso, valor_declarado)
        except ValueError as e:
            print(f"Error al crear paquete: {e}")
            return None
    
    def crear_paquete_express(self, descripcion, peso, valor_declarado, recargo=0.5):
        """Crea un paquete express"""
        try:
            return PaqueteExpress(descripcion, peso, valor_declarado, recargo)
        except ValueError as e:
            print(f"Error al crear paquete: {e}")
            return None
    
    def crear_envio(self, id_origen, id_destino, paquete):
        """Crea un nuevo envío"""
        cliente_origen = self.buscar_cliente(id_origen)
        cliente_destino = self.buscar_cliente(id_destino)
        
        if not cliente_origen:
            print(f"Cliente origen {id_origen} no encontrado")
            return None
        
        if not cliente_destino:
            print(f"Cliente destino {id_destino} no encontrado")
            return None
        
        try:
            envio = Envio(cliente_origen, cliente_destino, paquete)
            self.__envios.append(envio)
            print(f"Envío {envio.get_id_envio()} creado exitosamente")
            return envio
        except ValueError as e:
            print(f"Error al crear envío: {e}")
            return None
    
    def buscar_cliente(self, id_cliente):
        """Busca un cliente por ID"""
        for cliente in self.__clientes:
            if cliente.get_id() == id_cliente:
                return cliente
        return None
    
    def listar_clientes(self):
        """Lista todos los clientes registrados"""
        if not self.__clientes:
            print("No hay clientes registrados")
            return
        
        print("\n=== CLIENTES REGISTRADOS ===")
        for cliente in self.__clientes:
            print(f"{cliente}")
            print(f"  Dirección: {cliente.get_direccion()}")
            print(f"  Teléfono: {cliente.get_telefono()}")
            print()
    
    def listar_envios(self):
        """Lista todos los envíos (Polimorfismo)"""
        if not self.__envios:
            print("No hay envíos registrados")
            return
        
        print("\n=== LISTADO DE ENVÍOS ===")
        for envio in self.__envios:
            print(f"\n{envio}")
            print(f"  Origen: {envio.get_cliente_origen().get_nombre()}")
            print(f"  Destino: {envio.get_cliente_destino().get_nombre()}")
            print(f"  {envio.get_paquete()}")  # Polimorfismo aquí
            print(f"  Estado: {envio.get_paquete().get_estado()}")
            print(f"  Fecha: {envio.get_fecha_envio().strftime('%Y-%m-%d %H:%M')}")
    
    def reporte_ventas(self):
        """Genera reporte de ventas por tipo"""
        if not self.__envios:
            print("No hay envíos para generar reporte")
            return
        
        total_estandar = 0
        total_express = 0
        count_estandar = 0
        count_express = 0
        
        for envio in self.__envios:
            paquete = envio.get_paquete()
            precio = envio.get_precio_total()
            
            if isinstance(paquete, PaqueteEstandar):
                total_estandar += precio
                count_estandar += 1
            elif isinstance(paquete, PaqueteExpress):
                total_express += precio
                count_express += 1
        
        print("\n=== REPORTE DE VENTAS ===")
        print(f"Envíos Estándar: {count_estandar} - Total: ${total_estandar:.0f}")
        print(f"Envíos Express: {count_express} - Total: ${total_express:.0f}")
        print(f"TOTAL GENERAL: ${total_estandar + total_express:.0f}")
    
    def actualizar_estado_envio(self, id_envio, nuevo_estado):
        """Actualiza el estado de un envío específico"""
        for envio in self.__envios:
            if envio.get_id_envio() == id_envio:
                envio.actualizar_estado_paquete(nuevo_estado)
                return True
        print(f"Envío {id_envio} no encontrado")
        return False
    
    def guardar_datos(self, archivo="envios_data.json"):
        """Guarda los datos en archivo JSON"""
        datos = {
            "clientes": [],
            "envios": []
        }
        
        
        for cliente in self.__clientes:
            datos["clientes"].append({
                "id": cliente.get_id(),
                "nombre": cliente.get_nombre(),
                "telefono": cliente.get_telefono(),
                "email": cliente.get_email(),
                "direccion": {
                    "calle": cliente.get_direccion().get_calle(),
                    "ciudad": cliente.get_direccion().get_ciudad(),
                    "codigo_postal": cliente.get_direccion().get_codigo_postal()
                }
            })
        
        
        for envio in self.__envios:
            paquete = envio.get_paquete()
            datos["envios"].append({
                "id_envio": envio.get_id_envio(),
                "cliente_origen": envio.get_cliente_origen().get_id(),
                "cliente_destino": envio.get_cliente_destino().get_id(),
                "paquete": {
                    "tipo": paquete.get_tipo(),
                    "descripcion": paquete.get_descripcion(),
                    "peso": paquete.get_peso(),
                    "valor_declarado": paquete.get_valor_declarado(),
                    "estado": paquete.get_estado()
                },
                "precio_total": envio.get_precio_total(),
                "fecha_envio": envio.get_fecha_envio().isoformat()
            })
        
        try:
            with open(archivo, 'w', encoding='utf-8') as f:
                json.dump(datos, f, indent=2, ensure_ascii=False)
            print(f"Datos guardados en {archivo}")
        except Exception as e:
            print(f"Error al guardar: {e}")
    
    def get_clientes(self):
        return self.__clientes
    
    def get_envios(self):
        return self.__envios

def menu_principal():
    """Menú principal del sistema"""
    sistema = SistemaEnvios()
    
    while True:
        print("\n" + "="*50)
        print("    SISTEMA DE GESTIÓN DE ENVÍOS")
        print("="*50)
        print("1. Registrar cliente")
        print("2. Crear envío estándar")
        print("3. Crear envío express")
        print("4. Listar clientes")
        print("5. Listar envíos")
        print("6. Actualizar estado de envío")
        print("7. Reporte de ventas")
        print("8. Guardar datos")
        print("9. Salir")
        print("-"*50)
        
        try:
            opcion = input("Seleccione una opción (1-9): ")
            
            if opcion == "1":
                registrar_cliente_menu(sistema)
            elif opcion == "2":
                crear_envio_estandar_menu(sistema)
            elif opcion == "3":
                crear_envio_express_menu(sistema)
            elif opcion == "4":
                sistema.listar_clientes()
            elif opcion == "5":
                sistema.listar_envios()
            elif opcion == "6":
                actualizar_estado_menu(sistema)
            elif opcion == "7":
                sistema.reporte_ventas()
            elif opcion == "8":
                sistema.guardar_datos()
            elif opcion == "9":
                print("¡Gracias por usar el sistema!")
                break
            else:
                print("Opción inválida")
        
        except KeyboardInterrupt:
            print("\n¡Hasta luego!")
            break
        except Exception as e:
            print(f"Error: {e}")

def registrar_cliente_menu(sistema):
    """Menú para registrar cliente"""
    print("\n--- REGISTRAR CLIENTE ---")
    try:
        nombre = input("Nombre: ")
        telefono = input("Teléfono: ")
        email = input("Email: ")
        calle = input("Calle: ")
        ciudad = input("Ciudad: ")
        codigo_postal = input("Código postal: ")
        
        sistema.registrar_cliente(nombre, telefono, email, calle, ciudad, codigo_postal)
    except Exception as e:
        print(f"Error: {e}")

def crear_envio_estandar_menu(sistema):
    """Menú para crear envío estándar"""
    print("\n--- CREAR ENVÍO ESTÁNDAR ---")
    
    if not sistema.get_clientes():
        print("No hay clientes registrados. Registre clientes primero.")
        return
    
    try:
        sistema.listar_clientes()
        
        id_origen = input("ID cliente origen: ")
        id_destino = input("ID cliente destino: ")
        
        descripcion = input("Descripción del paquete: ")
        peso = float(input("Peso (kg): "))
        valor_declarado = float(input("Valor declarado: "))
        
        paquete = sistema.crear_paquete_estandar(descripcion, peso, valor_declarado)
        if paquete:
            sistema.crear_envio(id_origen, id_destino, paquete)
            
    except ValueError as e:
        print(f"Error en los datos: {e}")
    except Exception as e:
        print(f"Error: {e}")

def crear_envio_express_menu(sistema):
    """Menú para crear envío express"""
    print("\n--- CREAR ENVÍO EXPRESS ---")
    
    if not sistema.get_clientes():
        print("No hay clientes registrados. Registre clientes primero.")
        return
    
    try:
        sistema.listar_clientes()
        
        id_origen = input("ID cliente origen: ")
        id_destino = input("ID cliente destino: ")
        
        descripcion = input("Descripción del paquete: ")
        peso = float(input("Peso (kg): "))
        valor_declarado = float(input("Valor declarado: "))
        
        recargo_input = input("Recargo urgencia (0.5 por defecto): ")
        recargo = float(recargo_input) if recargo_input else 0.5
        
        paquete = sistema.crear_paquete_express(descripcion, peso, valor_declarado, recargo)
        if paquete:
            sistema.crear_envio(id_origen, id_destino, paquete)
            
    except ValueError as e:
        print(f"Error en los datos: {e}")
    except Exception as e:
        print(f"Error: {e}")

def actualizar_estado_menu(sistema):
    """Menú para actualizar estado de envío"""
    print("\n--- ACTUALIZAR ESTADO ---")
    
    if not sistema.get_envios():
        print("No hay envíos registrados.")
        return
    
    try:
        sistema.listar_envios()
        
        id_envio = input("ID del envío: ")
        print("Estados disponibles: pendiente, en_transito, entregado, devuelto")
        nuevo_estado = input("Nuevo estado: ")
        
        sistema.actualizar_estado_envio(id_envio, nuevo_estado)
        
    except Exception as e:
        print(f"Error: {e}")


print("=== DEMOSTRACIÓN DEL SISTEMA DE ENVÍOS ===\n")

sistema = SistemaEnvios()


print("1. REGISTRANDO CLIENTES:")
cliente1 = sistema.registrar_cliente("Ana García", "3001234567", "ana@email.com", 
                                   "Calle 123 #45-67", "Bogotá", "110111")

cliente2 = sistema.registrar_cliente("Carlos López", "3107654321", "carlos@email.com",
                                   "Carrera 89 #12-34", "Medellín", "050001")

cliente3 = sistema.registrar_cliente("María Rodríguez", "3209876543", "maria@email.com",
                                   "Avenida 456 #78-90", "Cali", "760001")


print("\n2. CREANDO PAQUETES Y ENVÍOS:")

paquete1 = sistema.crear_paquete_estandar("Documentos importantes", 0.5, 100000)
paquete2 = sistema.crear_paquete_express("Medicamentos urgentes", 1.2, 250000)
paquete3 = sistema.crear_paquete_estandar("Ropa", 2.5, 150000)


if paquete1:
    sistema.crear_envio("CLI001", "CLI002", paquete1)

if paquete2:
    sistema.crear_envio("CLI002", "CLI003", paquete2)

if paquete3:
    sistema.crear_envio("CLI003", "CLI001", paquete3)


print("\n3. LISTADO DE CLIENTES:")
sistema.listar_clientes()

print("\n4. LISTADO DE ENVÍOS (Polimorfismo):")
sistema.listar_envios()

print("\n5. ACTUALIZANDO ESTADOS:")
sistema.actualizar_estado_envio("ENV0001", "en_transito")
sistema.actualizar_estado_envio("ENV0002", "entregado")

print("\n6. REPORTE DE VENTAS:")
sistema.reporte_ventas()

print("\n7. GUARDANDO DATOS:")
sistema.guardar_datos()


print("\n=== SISTEMA COMPLETADO ===")
print("Para usar el menú interactivo, descomente la línea: menu_principal()")



=== DEMOSTRACIÓN DEL SISTEMA DE ENVÍOS ===

1. REGISTRANDO CLIENTES:
Cliente Ana García registrado exitosamente
Cliente Carlos López registrado exitosamente
Cliente María Rodríguez registrado exitosamente

2. CREANDO PAQUETES Y ENVÍOS:
Cliente destino CLI002 no encontrado
Cliente origen CLI002 no encontrado
Envío ENV0001 creado exitosamente

3. LISTADO DE CLIENTES:

=== CLIENTES REGISTRADOS ===
Cliente CLI001: Ana García (ana@email.com)
  Dirección: Calle 123 #45-67, Bogotá - 110111
  Teléfono: 3001234567

Cliente CLI003: Carlos López (carlos@email.com)
  Dirección: Carrera 89 #12-34, Medellín - 050001
  Teléfono: 3107654321

Cliente CLI005: María Rodríguez (maria@email.com)
  Dirección: Avenida 456 #78-90, Cali - 760001
  Teléfono: 3209876543


4. LISTADO DE ENVÍOS (Polimorfismo):

=== LISTADO DE ENVÍOS ===

Envío ENV0001: Carlos López → Ana García | $6500
  Origen: Carlos López
  Destino: Ana García
  Paquete Estándar: Ropa (2.5kg) - $6500
  Estado: pendiente
  Fecha: 2025-07-27 00:5